# Phase space grid example

This notebook demonstrates the **psgrid** workflow, an alternative to the Gaussian-ensemble
approach used in `example_1.ipynb`.

### The idea

For wavefront-aberration studies the quantity of interest is the **transfer function**
$p(\text{state} \mid x_0, v_0)$ — the probability of each output port given the atom's
initial phase-space coordinate. Because the simulation is deterministic for a single atom,
this requires **one atom per grid node**, not thousands.

Once the map is computed, any Gaussian cloud average follows analytically:
$$
P_s = \sum_{x_0, v_0} w(x_0, v_0)\, p(s \mid x_0, v_0) \, \Delta x_0 \, \Delta v_0
$$
without re-running ais++.

Storing **complex amplitudes** (not just probabilities) additionally lets you apply
arbitrary phase profiles offline — e.g. change the PSR shear wave-vector or scan the
interferometer phase — again without re-simulation.

### What this notebook covers

1. Run ais++ with `initmode psgrid` to produce a `_PSMAP.h5` file (625 atoms, ~1 s)
2. Load and inspect the raw output
3. Build a structured `(nx, nvx, n_paths)` array
4. Visualise the amplitude and phase maps
5. Sanity-check: probabilities sum to 1 at every grid node
6. Compute Gaussian-cloud-averaged port probabilities
7. Scan the interferometer phase **offline** by rotating complex amplitudes
8. Apply an offline phase shear and regenerate the PSR image

**Setup**
```bash
pip install numpy scipy matplotlib h5py jupyter

# Build and run the grid simulation (from the repo root)
cd examples/
ais++ -i input-files/PSR_EXAMPLE_PSGRID.aisi -o output-files/PSR_EXAMPLE_PSGRID.h5
```
The output is written to `output-files/PSR_EXAMPLE_PSGRID.h5`.

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import norm

try:
    plt.style.use('seaborn-v0_8-ticks')
except OSError:
    plt.style.use('seaborn-ticks')

plt.rcParams.update({
    'font.family'  : 'serif',
    'font.size'    : 9,
    'axes.labelsize': 9,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'legend.frameon': False,
    'savefig.dpi'  : 150,
})

## 1. Load the phase space map

In [ ]:
FNAME = 'output-files/PSR_EXAMPLE_PSGRID.h5'

with h5py.File(FNAME, 'r') as f:
    init_pos   = f['initial_positions'][:]   # (N_wps, 3)
    init_vel   = f['initial_velocities'][:]  # (N_wps, 3)
    final_pos  = f['final_positions'][:]     # (N_wps, 3)
    final_vel  = f['final_velocities'][:]    # (N_wps, 3)
    amp        = f['amplitudes'][:]          # (N_wps,)  |c|
    phase      = f['phases'][:]              # (N_wps,)  arg(c)
    phase_err  = f['phase_errors'][:]        # (N_wps,)  numerical error estimate
    state      = f['states'][:]              # (N_wps,)  0 or 1
    atom_idx   = f['atom_indices'][:]        # (N_wps,)  which atom
    paths_raw  = f['paths'][:]               # (N_wps,)  path strings

# Decode path strings (stored as variable-length bytes in HDF5)
paths = np.array([p.decode() if isinstance(p, bytes) else p for p in paths_raw])

# Complex amplitude
c = amp * np.exp(1j * phase)

n_wps   = len(amp)
n_atoms = int(atom_idx.max()) + 1
unique_paths = list(dict.fromkeys(paths))   # preserve encounter order
n_paths = len(unique_paths)

print(f'Total wavepackets : {n_wps:,}')
print(f'Atoms (grid nodes): {n_atoms:,}')
print(f'Paths per atom    : {n_paths}  →  {unique_paths}')
print(f'Max phase error   : {phase_err.max():.2e} rad')

## 2. Build structured arrays

Pivot the flat wavepacket list into `(n_atoms, n_paths)` arrays, then reshape to
`(nx, nvx, n_paths)` using the known grid ordering (x outermost, vx innermost).

In [ ]:
path_to_idx = {p: i for i, p in enumerate(unique_paths)}

# Allocate
C      = np.zeros((n_atoms, n_paths), dtype=complex)  # complex amplitude
ST     = np.zeros((n_atoms, n_paths), dtype=int)       # state
XF     = np.zeros((n_atoms, n_paths))                  # final x
VXF    = np.zeros((n_atoms, n_paths))                  # final vx

for i in range(n_wps):
    ai = atom_idx[i]
    pi = path_to_idx[paths[i]]
    C[ai, pi]   = c[i]
    ST[ai, pi]  = state[i]
    XF[ai, pi]  = final_pos[i, 0]
    VXF[ai, pi] = final_vel[i, 0]

# Initial conditions — same for all paths of a given atom; read from first occurrence
first = np.array([np.searchsorted(atom_idx, ai) for ai in range(n_atoms)])
X0  = init_pos[first, 0]   # (n_atoms,)
VX0 = init_vel[first, 0]   # (n_atoms,)

# Infer grid shape from unique sorted values
xs  = np.unique(X0)
vxs = np.unique(VX0)
nx, nvx = len(xs), len(vxs)
print(f'Grid shape: {nx} x {nvx}  '
      f'(x: {xs[0]*1e3:.2f} to {xs[-1]*1e3:.2f} mm, '
      f'vx: {vxs[0]*1e3:.2f} to {vxs[-1]*1e3:.2f} mm/s)')

# Reshape to (nx, nvx, n_paths)  — grid constructor loops x outermost, vx innermost
C_grid   = C.reshape(nx, nvx, n_paths)
ST_grid  = ST.reshape(nx, nvx, n_paths)
XF_grid  = XF.reshape(nx, nvx, n_paths)
VXF_grid = VXF.reshape(nx, nvx, n_paths)
X0_grid  = X0.reshape(nx, nvx)
VX0_grid = VX0.reshape(nx, nvx)

## 3. Amplitude and phase maps

For each of the four PSR ports, plot $|c(x_0, v_{x0})|^2$ (transition probability)
and $\arg c(x_0, v_{x0})$ (accumulated phase, relative to the grid mean).
The phase variation across the grid encodes wavefront aberrations.

In [ ]:
extent = [vxs[0]*1e3, vxs[-1]*1e3, xs[0]*1e6, xs[-1]*1e6]  # mm/s, um

fig, axes = plt.subplots(2, n_paths, figsize=(9, 5),
                         gridspec_kw=dict(hspace=0.35, wspace=0.35))

for pi, pth in enumerate(unique_paths):
    prob  = np.abs(C_grid[:, :, pi])**2
    # Phase relative to grid mean to remove the huge absolute offset
    ph    = np.angle(C_grid[:, :, pi] * np.exp(-1j * np.angle(C_grid[:, :, pi]).mean()))
    st_label = f"state {ST_grid[0, 0, pi]}"

    im0 = axes[0, pi].imshow(prob, origin='lower', extent=extent,
                              aspect='auto', cmap='viridis', vmin=0, vmax=0.5)
    axes[0, pi].set_title(f'path {pth}\n({st_label})', fontsize=8)
    axes[0, pi].set_xlabel(r'$v_{x0}$ [mm/s]')
    if pi == 0:
        axes[0, pi].set_ylabel(r'$x_0$ [$\mu$m]')
    plt.colorbar(im0, ax=axes[0, pi], label=r'$|c|^2$')

    im1 = axes[1, pi].imshow(ph, origin='lower', extent=extent,
                              aspect='auto', cmap='RdBu', vmin=-np.pi, vmax=np.pi)
    axes[1, pi].set_xlabel(r'$v_{x0}$ [mm/s]')
    if pi == 0:
        axes[1, pi].set_ylabel(r'$x_0$ [$\mu$m]')
    plt.colorbar(im1, ax=axes[1, pi], label=r'$\arg(c)$ [rad]')

axes[0, 0].set_title(f'$|c|^2$ maps', loc='left', fontsize=9, fontweight='bold')
axes[1, 0].set_title(r'$\arg(c)$ maps (relative)', loc='left', fontsize=9, fontweight='bold')
plt.suptitle('Phase space transfer functions — PSR MZ ($n=1$, $T=2.225$ s)', y=1.01)
plt.savefig('psmap_amplitude_phase.png', bbox_inches='tight')
plt.show()

## 4. Sanity check: probabilities sum to 1

In [ ]:
total_prob = np.sum(np.abs(C_grid)**2, axis=2)  # (nx, nvx)

print(f'Sum of |c|² over all paths:')
print(f'  mean  = {total_prob.mean():.6f}')
print(f'  min   = {total_prob.min():.6f}')
print(f'  max   = {total_prob.max():.6f}')
print(f'  std   = {total_prob.std():.2e}')

assert np.allclose(total_prob, 1.0, atol=1e-4), \
    'Probability normalisation violated — check amplitude threshold or path selection.'

## 5. Gaussian-cloud-averaged port probabilities

Define a Gaussian cloud $(x_{\rm cm}, v_{x,\rm cm}, \sigma_x, \sigma_{v_x})$ and compute
the mean probability of each port as a weighted sum over the grid:
$$
P_j = \frac{\sum_{x_0, v_0} w(x_0, v_0)\,|c_j(x_0, v_0)|^2}{\sum w}
$$
This replaces what previously required $\sim 10^6$-atom simulations.

In [ ]:
# Physical constants
kB       = 1.380649e-23       # J/K
m_Sr87   = 86.909 * 1.66054e-27  # kg

def gaussian_weights(x0_grid, vx0_grid, x_cm, vx_cm, sigma_x, sigma_vx):
    """Normalised 2D Gaussian weights over the phase space grid."""
    w = (np.exp(-0.5 * ((x0_grid  - x_cm ) / sigma_x )**2) *
         np.exp(-0.5 * ((vx0_grid - vx_cm) / sigma_vx)**2))
    return w / w.sum()

def cloud_port_probs(C_grid, x0_grid, vx0_grid, x_cm=0., vx_cm=0.,
                     sigma_x=1e-4, T_trans=1e-9):
    """Mean probability of each port for a Gaussian cloud."""
    sigma_vx = np.sqrt(kB * T_trans / m_Sr87)
    w = gaussian_weights(x0_grid, vx0_grid, x_cm, vx_cm, sigma_x, sigma_vx)
    # w: (nx, nvx),  |C_grid|²: (nx, nvx, n_paths)
    P = np.einsum('ij,ijp->p', w, np.abs(C_grid)**2)
    return P

P = cloud_port_probs(C_grid, X0_grid, VX0_grid)

print('Port probabilities for reference cloud (σ_x=100 µm, T=1 nK, centred):')
for pth, prob in zip(unique_paths, P):
    print(f'  path {pth}  →  P = {prob:.4f}')
print(f'  sum           = {P.sum():.6f}')

## 6. Interferometer phase scan — offline

Applying a global phase offset $\delta\phi$ to one interferometer arm (here the upper arm,
paths starting with `01`) rotates the complex amplitude of those paths and traces out the
interference fringe — **without re-running ais++**.

In the PSR geometry, the spatially resolved fringe arises because atoms at different
$(x_0, v_{x0})$ accumulate different phases through the Gaussian beam wavefronts.
The mean state-0 population is:
$$
P_{s=0}(\delta\phi) = \sum_{x_0,v_0} w(x_0, v_0)
  \sum_{j:\,s_j=0} |c_j(x_0, v_0)|^2
$$
where the upper-arm paths have their phase shifted by $\delta\phi$.

> Note: in PSR mode the four output ports are spatially separated at detection, so
> the fringe is read from the transverse *position* distribution rather than the integrated
> population (see §8). The scan below illustrates the principle.

In [ ]:
# Identify upper-arm paths (second character = '1', i.e. atom flipped after first π/2)
upper_arm = np.array([1 if pth[1] == '1' else 0 for pth in unique_paths], dtype=float)
print('Upper arm mask:', dict(zip(unique_paths, upper_arm.astype(int))))

phi_scan = np.linspace(0, 2*np.pi, 200)

sigma_vx_ref = np.sqrt(kB * 1e-9 / m_Sr87)
w = gaussian_weights(X0_grid, VX0_grid, 0., 0., 1e-4, sigma_vx_ref)  # (nx, nvx)

P0_scan = np.zeros(len(phi_scan))   # mean state-0 prob vs phi
P1_scan = np.zeros(len(phi_scan))   # mean state-1 prob vs phi

for k, dphi in enumerate(phi_scan):
    # Rotate upper-arm amplitudes by exp(i * dphi)
    phase_rot = np.exp(1j * dphi * upper_arm)   # (n_paths,)
    C_rot = C_grid * phase_rot[np.newaxis, np.newaxis, :]  # (nx, nvx, n_paths)

    # State-0 probability at each grid node = sum |c_j|² for state-0 paths
    # (paths are spatially separated in PSR; no amplitude-level interference between them)
    s0_mask = (ST_grid[0, 0, :] == 0)   # which paths end in state 0
    s1_mask = ~s0_mask

    prob0 = np.sum(np.abs(C_rot[:, :, s0_mask])**2, axis=2)  # (nx, nvx)
    prob1 = np.sum(np.abs(C_rot[:, :, s1_mask])**2, axis=2)

    P0_scan[k] = np.sum(w * prob0)
    P1_scan[k] = np.sum(w * prob1)

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(phi_scan / np.pi, P0_scan, label=r'$P_{s=0}$')
ax.plot(phi_scan / np.pi, P1_scan, label=r'$P_{s=1}$', ls='--')
ax.set_xlabel(r'Upper-arm phase $\delta\phi / \pi$')
ax.set_ylabel('Mean port probability')
ax.set_title('Offline phase scan (Gaussian cloud, $\\sigma_x=100\\,\\mu$m, $T=1\\,\\rm nK$)')
ax.legend()
ax.set_xlim(0, 2)
plt.tight_layout()
plt.savefig('psmap_phase_scan.png', bbox_inches='tight')
plt.show()

# Fringe contrast from fit
contrast = (P0_scan.max() - P0_scan.min()) / (P0_scan.max() + P0_scan.min())
print(f'Fringe contrast: {contrast:.4f}')

## 7. Offline phase shear — regenerate the PSR image

In the PSR readout, the last pulse applies a transverse kick $\hbar k_{x,\rm PSR}$ to one
arm, imprinting a spatial phase $k_{x,\rm PSR} x_{\rm final}$ on the atoms.
Because the complex amplitudes are stored, we can **change the effective shear
wave-vector offline** by multiplying each wavepacket by
$\exp(i\,\delta k_x\, x_{\rm final})$ and recomputing the weighted atom density.

This is useful for optimising the shear, scanning it, or exploring non-standard shear
geometries, all from a single ais++ run.

In [ ]:
def psr_image(C_grid, XF_grid, X0_grid, VX0_grid, ST_grid,
              kappa_extra=0., x_cm=0., vx_cm=0.,
              sigma_x=1e-4, T_trans=1e-9,
              bins=60, x_range=(-0.006, 0.006)):
    """
    Reconstruct the PSR x-histogram for a given cloud and extra shear.

    kappa_extra  : additional shear in rad/m applied offline
                   (on top of the kappa_PSR=3140 rad/m already in the simulation)
    """
    sigma_vx = np.sqrt(kB * T_trans / m_Sr87)
    w = gaussian_weights(X0_grid, VX0_grid, x_cm, vx_cm, sigma_x, sigma_vx)  # (nx, nvx)

    # Apply extra phase shear: multiply each wavepacket by exp(i * kappa_extra * x_final)
    phase_shift = np.exp(1j * kappa_extra * XF_grid)   # (nx, nvx, n_paths)
    C_sheared   = C_grid * phase_shift

    # Weight for each wavepacket in the histogram:
    # Gaussian(x0,vx0) * |c_sheared|^2
    wt = w[:, :, np.newaxis] * np.abs(C_sheared)**2   # (nx, nvx, n_paths)

    # Flatten to get (x_final, weight) pairs for all wavepackets
    xf_flat = XF_grid.ravel()           # repeated n_paths times? No — XF_grid is (nx,nvx,np)
    # XF_grid has shape (nx, nvx, n_paths), flatten all three axes
    xf_flat = XF_grid.reshape(-1)
    wt_flat = wt.reshape(-1)

    hist, edges = np.histogram(xf_flat, bins=bins, range=x_range, weights=wt_flat)
    xc = 0.5 * (edges[:-1] + edges[1:])
    return xc, hist


kappa_extra_vals = [0, 1000, 3140]   # rad/m — 0 = unchanged, 3140 = double the shear
labels = [r'$\delta\kappa = 0$  (original)',
          r'$\delta\kappa = +1000\,\rm rad/m$',
          r'$\delta\kappa = +3140\,\rm rad/m$  (doubled)']

fig, axes = plt.subplots(len(kappa_extra_vals), 1, figsize=(5.5, 5),
                          sharex=True, gridspec_kw=dict(hspace=0.12))

for ax, kex, lbl in zip(axes, kappa_extra_vals, labels):
    xc, hist = psr_image(C_grid, XF_grid, X0_grid, VX0_grid, ST_grid, kappa_extra=kex)
    bw = (xc[1] - xc[0]) * 1e3
    ax.bar(xc * 1e3, hist, width=bw, color='#4daf4a', alpha=0.75, linewidth=0)
    ax.set_ylabel('Weighted atoms')
    ax.annotate(lbl, xy=(0.02, 0.88), xycoords='axes fraction', fontsize=8)

axes[-1].set_xlabel(r'Final $x$ position [mm]')
plt.suptitle('PSR image for different offline shear values\n'
             r'(base shear $\kappa_{\rm PSR}=3140\,\rm rad/m$)', y=1.01)
plt.savefig('psmap_shear_scan.png', bbox_inches='tight')
plt.show()

## 8. Cloud-parameter scan without re-simulation

Scan the cloud centre $x_{\rm cm}$ over one fringe period and compute the state-0
population for each position. This is the central benefit of the grid approach:
exploring cloud-parameter space costs only a weighted sum, not a new ais++ run.

In [ ]:
x_cm_scan = np.linspace(-3e-4, 3e-4, 80)   # ±3 σ_x
P0_xcm    = np.zeros(len(x_cm_scan))

s0_mask = (ST_grid[0, 0, :] == 0)

for k, xcm in enumerate(x_cm_scan):
    P = cloud_port_probs(C_grid, X0_grid, VX0_grid, x_cm=xcm)
    P0_xcm[k] = P[s0_mask].sum()

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(x_cm_scan * 1e6, P0_xcm)
ax.set_xlabel(r'Cloud centre $x_{\rm cm}$ [$\mu$m]')
ax.set_ylabel(r'$P_{s=0}$')
ax.set_title(r'State-0 probability vs cloud position — no re-simulation')
plt.tight_layout()
plt.savefig('psmap_xcm_scan.png', bbox_inches='tight')
plt.show()